In [36]:
import os
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [37]:
df = pd.read_csv("/data/bbg/datasets/intogen/output/collaborations/20260324_BLCA_EPorta/intogen_output/highrisk_lowrisk_mibc/20260320/steps/oncodrive3d/OTHERS_WXS_BLADDER_EPORTA_HIGHRISK_NMIBC_2026.o3d_genes.tsv", sep='\t')

In [38]:
df.columns

Index(['Gene', 'Uniprot_ID', 'pval', 'qval', 'C_gene', 'C_pos', 'C_label',
       'Pos_top_vol', 'Score_obs_sim_top_vol', 'Mut_in_gene', 'Clust_mut',
       'Clust_res', 'Mut_in_top_vol', 'Mut_in_top_cl_vol', 'PAE_top_vol',
       'pLDDT_top_vol', 'pLDDT_top_cl_vol', 'Ratio_not_in_structure',
       'Ratio_WT_mismatch', 'Mut_zero_mut_prob', 'Pos_zero_mut_prob', 'Cancer',
       'Cohort', 'F', 'Transcript_ID', 'O3D_transcript_ID',
       'Transcript_status', 'Status'],
      dtype='object')

In [39]:
# translate the list of residue coordinates into a list of genomic coordinates

INTOGEN_DATASETS = '/data/bbg/datasets/intogen/output/collaborations/20260324_BLCA_EPorta/intogen_output/highrisk_lowrisk_mibc/20260320/'
sat_datasets = os.path.join(INTOGEN_DATASETS, 'steps/boostDM/saturation')
sat_datasets

COHORTS_PATH = os.path.join(INTOGEN_DATASETS, 'cohorts.tsv')

In [40]:
def load_ttypes_map(file):

    df = pd.read_csv(file, sep='\t')
    df = df[['COHORT', 'CANCER_TYPE']]
    return df.set_index('COHORT').to_dict()['CANCER_TYPE']

ttype_map = load_ttypes_map(COHORTS_PATH)

In [62]:
def residue_to_genomic(df):

    df['CANCER_TYPE'] = df['Cohort'].map(ttype_map)
    df = df[df['Transcript_status'] != 'O3D_missing']
    dh = {'HUGO Symbol': [], 'chromosome': [], 'genomic position': [], 'CANCER_TYPE': []}
    dg = df[df['C_gene'] == 1].copy()
    for gene, ttype in dg.groupby(['Gene', 'CANCER_TYPE']).size().index:
        try:
            residues = dg[(dg['Gene'] == gene) & (dg['CANCER_TYPE'] == ttype)]['C_pos'].values[0]
            residues = list(map(int, residues.strip(" []").split()))
            saturation = pd.read_csv(f"{sat_datasets}/{gene}.vep.gz", sep='\t')
            saturation = saturation[saturation['Protein_position'] != '-']
            saturation['Protein_position'] = saturation['Protein_position'].astype(int)
            genomic_coordinates = saturation[saturation['Protein_position'].isin(residues)][['Chromosome', 'Position']].drop_duplicates()
            dh['HUGO Symbol'].extend([gene])
            dh['chromosome'].extend(['chr' + str(genomic_coordinates['Chromosome'].iloc[0])])
            dh['genomic position'].extend([','.join(map(str, genomic_coordinates['Position'].values))])
            dh['CANCER_TYPE'].extend([ttype])
        except FileNotFoundError:
            print(f"Saturation file for gene {gene} not found.")
            continue
    dh = pd.DataFrame(dh)
    return dh

In [63]:
#df.head()
dh = residue_to_genomic(df)

Saturation file for gene ACTB not found.
Saturation file for gene ADCY8 not found.
Saturation file for gene ADGRV1 not found.
Saturation file for gene AHR not found.
Saturation file for gene CCDC138 not found.
Saturation file for gene CHRD not found.
Saturation file for gene CLEC4M not found.
Saturation file for gene DIAPH2 not found.
Saturation file for gene EPG5 not found.
Saturation file for gene ERICH3 not found.
Saturation file for gene ERICH6B not found.
Saturation file for gene FHL3 not found.
Saturation file for gene FOXD4L6 not found.
Saturation file for gene FRG1 not found.
Saturation file for gene H2BC12L not found.
Saturation file for gene HS6ST3 not found.
Saturation file for gene IRF5 not found.
Saturation file for gene KLF18 not found.
Saturation file for gene MADCAM1 not found.
Saturation file for gene MROH2B not found.
Saturation file for gene NANOGP8 not found.
Saturation file for gene NCAN not found.
Saturation file for gene NPEPPS not found.
Saturation file for gene

In [64]:
dh

,HUGO Symbol,chromosome,genomic position,CANCER_TYPE
0,AFDN,chr6,"167889230,167889231,167889232,167951903,167951...",HIGHRISK_NMIBC
1,ERBB2,chr17,"39711939,39711940,39711941,39711954,39711955,3...",HIGHRISK_NMIBC
2,ERBB3,chr12,"56085001,56085002,56085003,56085004,56085005,5...",HIGHRISK_NMIBC
3,ERCC2,chr19,"45352527,45352528,45352529,45352557,45352558,4...",HIGHRISK_NMIBC
4,FBXW7,chr4,"152324284,152324285,152324286,152326006,152326...",HIGHRISK_NMIBC
5,FGFR3,chr4,"1801837,1801838,1801839,1801840,1801841,180184...",HIGHRISK_NMIBC
6,FUT4,chr11,"94544737,94544738,94544739",HIGHRISK_NMIBC
7,GNA13,chr17,"65014791,65014792,65014793",HIGHRISK_NMIBC
8,HRAS,chr11,"533873,533874,533875,534284,534285,534286,5342...",HIGHRISK_NMIBC
9,KRAS,chr12,"25245346,25245347,25245348,25245349,25245350,2...",HIGHRISK_NMIBC


In [70]:
import numpy as np
import pandas as pd

def mimic_explode(s):
    """Mimics the behavior of Series.explode() for older pandas versions."""
    # Get the length of each list to know how many times to repeat the index
    # (fillna(1) ensures that rows with NaN/missing values aren't accidentally deleted)
    lens = s.str.len().fillna(1).astype(int)
    
    # Flatten the lists into a single 1D array
    # If a value isn't a list (like NaN), wrap it in a list so it flattens correctly
    flat_data = [item for sublist in s for item in (sublist if isinstance(sublist, list) else [sublist])]
    
    # Rebuild the Series with the repeated index
    return pd.Series(flat_data, index=s.index.repeat(lens))

result = (dh.set_index(['HUGO Symbol', 'CANCER_TYPE'])
            .apply(lambda x: mimic_explode(x.str.split(',')))
            .reset_index())

result

,HUGO Symbol,CANCER_TYPE,chromosome,genomic position
0,AFDN,HIGHRISK_NMIBC,chr6,167889230
1,AFDN,HIGHRISK_NMIBC,chr6,167889231
2,AFDN,HIGHRISK_NMIBC,chr6,167889232
3,AFDN,HIGHRISK_NMIBC,chr6,167951903
4,AFDN,HIGHRISK_NMIBC,chr6,167951904
5,AFDN,HIGHRISK_NMIBC,chr6,167951905
6,ERBB2,HIGHRISK_NMIBC,chr17,39711939
7,ERBB2,HIGHRISK_NMIBC,chr17,39711940
8,ERBB2,HIGHRISK_NMIBC,chr17,39711941
9,ERBB2,HIGHRISK_NMIBC,chr17,39711954


# Emulate pipeline

In [73]:
def mimic_explode(s):
    
    """Mimics the behavior of Series.explode() for older pandas versions."""
    # Get the length of each list to know how many times to repeat the index
    # (fillna(1) ensures that rows with NaN/missing values aren't accidentally deleted)
    lens = s.str.len().fillna(1).astype(int)
    
    # Flatten the lists into a single 1D array
    # If a value isn't a list (like NaN), wrap it in a list so it flattens correctly
    flat_data = [item for sublist in s for item in (sublist if isinstance(sublist, list) else [sublist])]
    
    # Rebuild the Series with the repeated index
    return pd.Series(flat_data, index=s.index.repeat(lens))


def residue_to_genomic(df):

    """
    From an input file with the significant genes and residues from oncodrive3d output of type *.o3d_genes.tsv, 
    this function generates a dataframe with the genomic coordinates of significantly 3D clustered residues.
    """

    dg = df.copy()
    
    dh = {'HUGO Symbol': [], 'chromosome': [], 'genomic position': [], 'CANCER_TYPE': []}
    
    for gene, ttype in dg.groupby(['Gene', 'CANCER_TYPE']).size().index:
        try:
            residues = dg[(dg['Gene'] == gene) & (dg['CANCER_TYPE'] == ttype)]['C_pos'].values[0]
            residues = list(map(int, residues.strip(" []").split()))
            saturation = pd.read_csv(f"{sat_datasets}/{gene}.vep.gz", sep='\t')
            saturation['Protein_position'] = saturation['Protein_position'].astype(str)
            saturation = saturation[saturation['Protein_position'] != '-']
            saturation['Protein_position'] = saturation['Protein_position'].astype(int)
            genomic_coordinates = saturation[saturation['Protein_position'].isin(residues)][['Chromosome', 'Position']].drop_duplicates()
            dh['HUGO Symbol'].extend([gene])
            dh['chromosome'].extend(['chr' + str(genomic_coordinates['Chromosome'].iloc[0])])
            dh['genomic position'].extend([','.join(map(str, genomic_coordinates['Position'].values))])
            dh['CANCER_TYPE'].extend([ttype])
        except FileNotFoundError:
            print(f"Saturation file for gene {gene} not found.")
            continue
    dh = pd.DataFrame(dh)
    return dh


def expand_genomic_coordinates(df):
    
    """
    From a dataframe with the genomic coordinates of significantly 3D clustered residues
    per gene and tumor type, as generated by the function residue_to_genomic, 
    this function generates a dataframe with one row per genomic coordinate.
    """

    dk = (df.set_index(['HUGO Symbol', 'CANCER_TYPE'])
            .apply(lambda x: mimic_explode(x.str.split(',')))
            .reset_index())
    dk.rename(columns={'genomic position': 'pos'}, inplace=True)
    dk = dk[['chromosome', 'pos', 'CANCER_TYPE']]
    return dk

In [74]:
df = pd.read_csv('/data/bbg/datasets/boostdm_runs/boostdm-pipeline-bladder-2026/work/62/ed52bc7aa3f1b5c5cc673302da0ed3/oncodrive3d_raw.tsv.gz', sep='\t')
df

,Gene,C_gene,C_pos,CANCER_TYPE
0,FGFR3,1,[248 249 371 370 369 373 368],LOWRISK_NMIBC
1,PIK3CA,1,[ 545 546 542 1047 1043 515],LOWRISK_NMIBC
2,HRAS,1,[61 12 13],LOWRISK_NMIBC
3,CCDC138,1,[473 472],LOWRISK_NMIBC
4,CELSR2,1,[16 17],LOWRISK_NMIBC
5,AKT1,1,[17],LOWRISK_NMIBC
6,CLEC4M,1,[210],LOWRISK_NMIBC
7,PTGER4,1,[385 381],LOWRISK_NMIBC
8,ERICH3,1,[1019 1022],LOWRISK_NMIBC
9,GNA13,1,[200],LOWRISK_NMIBC


In [75]:
dh = residue_to_genomic(df)

Saturation file for gene ABCD1 not found.
Saturation file for gene ABCF1 not found.
Saturation file for gene ACSS3 not found.
Saturation file for gene ACTB not found.
Saturation file for gene ADCY8 not found.
Saturation file for gene ADCY8 not found.
Saturation file for gene ADGRV1 not found.
Saturation file for gene AHR not found.
Saturation file for gene AHR not found.
Saturation file for gene AK2 not found.
Saturation file for gene ANKHD1 not found.
Saturation file for gene ANP32E not found.
Saturation file for gene APOBR not found.
Saturation file for gene ARSD not found.
Saturation file for gene ATXN1 not found.
Saturation file for gene C12orf43 not found.
Saturation file for gene C3orf70 not found.
Saturation file for gene CABLES1 not found.
Saturation file for gene CACNA1E not found.
Saturation file for gene CCDC138 not found.
Saturation file for gene CCDC138 not found.
Saturation file for gene CELSR2 not found.
Saturation file for gene CHRD not found.
Saturation file for gene C

In [76]:
dh

,HUGO Symbol,chromosome,genomic position,CANCER_TYPE
0,AFDN,chr6,"167889230,167889231,167889232,167951903,167951...",HIGHRISK_NMIBC
1,AFDN,chr6,"167951903,167951904,167951905",LOWRISK_NMIBC
2,AKT1,chr14,"104780212,104780213,104780214",LOWRISK_NMIBC
3,ARHGAP5,chr14,"32092134,32092135,32092136",MIBC
4,ASXL2,chr2,"25756052,25756053,25756054,25756064,25756065,2...",MIBC
5,BAP1,chr3,"52407184,52407185,52407186,52407202,52407203,5...",MIBC
6,BMP6,chr6,"7727307,7727308,7727309",MIBC
7,BRAF,chr7,"140753335,140753336,140753337,140753347,140753...",MIBC
8,CREBBP,chr16,"3731303,3731304,3731305,3731372,3731373,373137...",MIBC
9,DEK,chr6,"18263865,18263866,18263867",MIBC


In [80]:
# expand genomic coordinates

# Assuming 'dh' is your dataframe and the columns you want to split are in a list
# For this example, let's pretend you are splitting a column called 'VALUES'
cols_to_split = ['genomic position'] # Update this to the actual column(s) containing the commas

# 1. Split the strings into lists for the target columns
split_series = dh[cols_to_split[0]].astype(str).str.split(',')

# 2. Figure out how many items are in each list (this is our multiplier)
lens = split_series.str.len().fillna(1).astype(int)

# 3. Repeat the ENTIRE dataframe's rows based on those lengths
result_df = dh.loc[dh.index.repeat(lens)].copy()

# 4. Flatten the lists and overwrite the columns in the repeated dataframe
for col in cols_to_split:
    # Split the column
    s = dh[col].astype(str).str.split(',')
    # Flatten it
    flat_data = [item for sublist in s for item in (sublist if isinstance(sublist, list) else [sublist])]
    # Assign it back
    result_df[col] = flat_data

result_df

,HUGO Symbol,chromosome,genomic position,CANCER_TYPE
0,AFDN,chr6,167889230,HIGHRISK_NMIBC
0,AFDN,chr6,167889231,HIGHRISK_NMIBC
0,AFDN,chr6,167889232,HIGHRISK_NMIBC
0,AFDN,chr6,167951903,HIGHRISK_NMIBC
0,AFDN,chr6,167951904,HIGHRISK_NMIBC
0,AFDN,chr6,167951905,HIGHRISK_NMIBC
1,AFDN,chr6,167951903,LOWRISK_NMIBC
1,AFDN,chr6,167951904,LOWRISK_NMIBC
1,AFDN,chr6,167951905,LOWRISK_NMIBC
2,AKT1,chr14,104780212,LOWRISK_NMIBC


In [78]:
dk

,chromosome,pos,CANCER_TYPE
0,chr6,167889230,HIGHRISK_NMIBC
1,chr6,167889231,HIGHRISK_NMIBC
2,chr6,167889232,HIGHRISK_NMIBC
3,chr6,167951903,HIGHRISK_NMIBC
4,chr6,167951904,HIGHRISK_NMIBC
5,chr6,167951905,HIGHRISK_NMIBC
6,chr6,167951903,LOWRISK_NMIBC
7,chr6,167951904,LOWRISK_NMIBC
8,chr6,167951905,LOWRISK_NMIBC
9,chr14,104780212,LOWRISK_NMIBC


In [82]:
result_df.shape, dk.shape

((1242, 4), (1242, 3))